# Puma to CZ2020

In [15]:
import os
import pandas as pd
import numpy as np 

In [16]:
os.chdir("/Users/mengsong/Library/CloudStorage/Dropbox/research/wfh/cz/cz20")

# Puma2020 to CZ2020

In [ ]:
puma2020 = pd.read_csv('data/geocorr2022_2601502573.csv', encoding="latin-1", skiprows=[1],dtype=str)
puma2020 = puma2020[puma2020['puma22'] != ' ']

puma2020["afact"] = puma2020["afact"].astype(float)
puma2020.groupby(['state','puma22'])['afact'].sum().describe()

count    2486.00000
mean        1.00000
std         0.00003
min         0.99970
25%         1.00000
50%         1.00000
75%         1.00000
max         1.00020
Name: afact, dtype: float64

In [ ]:
# https://www.ers.usda.gov/data-products/commuting-zones-and-labor-market-areas
county20_cz20 = pd.read_csv('data/commuting-zones-2020.csv')

# Create standardized county identifiers
puma2020['county_fips'] = puma2020['county']
county20_cz20['county_fips'] = county20_cz20['FIPStxt'].astype(str).str.zfill(5)

# Check
print("County code lengths:")
print(f"  PUMA2020 county_fips: {puma2020['county_fips'].str.len().unique()}")
print(f"  County20-CZ20 county_fips: {county20_cz20['county_fips'].str.len().unique()}")

print(f"\nUnique county counts:")
print(f"  PUMA2020: {puma2020['county_fips'].nunique()}")
print(f"  County20-CZ20: {county20_cz20['county_fips'].nunique()}")

# Check overlap
puma2020_counties = set(puma2020['county_fips'].unique())
county20_counties = set(county20_cz20['county_fips'].unique())

overlap = len(puma2020_counties & county20_counties)
print(f'\nOverlap: {overlap} counties in both datasets')
print(f"  Overlap rate (of PUMA2020): {100*overlap/len(puma2020_counties):.2f}%")
print(f"  Overlap rate (of County20-CZ20): {100*overlap/len(county20_counties):.2f}%")

# Check for any mismatches
only_in_puma2020 = puma2020_counties - county20_counties
only_in_county20 = county20_counties - puma2020_counties

if len(only_in_puma2020) > 0:
    print(f'\nCounties only in PUMA2020: {len(only_in_puma2020)}')
    if len(only_in_puma2020) <= 20:
        print(sorted(only_in_puma2020))
    else:
        print(sorted(only_in_puma2020)[:20], '...')
    
if len(only_in_county20) > 0:
    print(f'\nCounties only in County20-CZ20: {len(only_in_county20)}')
    if len(only_in_county20) <= 20:
        print(sorted(only_in_county20))
    else:
        print(sorted(only_in_county20)[:20], '...')

County code lengths:
  PUMA2020 county_fips: [5]
  County20-CZ20 county_fips: [5]

Unique county counts:
  PUMA2020: 3222
  County20-CZ20: 3222

Overlap: 3222 counties in both datasets
  Overlap rate (of PUMA2020): 100.00%
  Overlap rate (of County20-CZ20): 100.00%


In [30]:
# Merge PUMA2020 with County20-CZ20
puma2020_cz20 = pd.merge(
    puma2020,
    county20_cz20,
    how='left',
    left_on='county_fips',
    right_on='county_fips'
)

print(f'Null CZ codes: {puma2020_cz20["CZ2020"].isnull().sum()}')

Null CZ codes: 0


In [20]:
# Construct allocation factor: fraction of PUMA population allocated to each CZ
puma2020_cz20_crosswalk = (
    puma2020_cz20.groupby(['state', 'puma22', 'CZ2020'])
    .agg({'afact': 'sum'})
    .reset_index()
)

# Create combined PUMA 2020 code (state + puma, 7 digits total)
puma2020_cz20_crosswalk['puma2020'] = (
    puma2020_cz20_crosswalk['state'].astype(int).astype(str).str.zfill(2) + 
    puma2020_cz20_crosswalk['puma22'].astype(int).astype(str).str.zfill(5)
)

# Rename and select final columns
puma2020_cz20_crosswalk = puma2020_cz20_crosswalk.rename(columns={'CZ2020': 'cz2020', 'afact': 'afactor'})
puma2020_cz20_crosswalk = puma2020_cz20_crosswalk[['puma2020', 'cz2020', 'afactor']]
puma2020_cz20_crosswalk['afactor'] = puma2020_cz20_crosswalk['afactor'].round(3)

# Verify allocation factors sum to 1 for each PUMA 2020
check = puma2020_cz20_crosswalk.groupby('puma2020')['afactor'].sum()
print(check.describe())

count    2486.000000
mean        0.999996
std         0.000162
min         0.998000
25%         1.000000
50%         1.000000
75%         1.000000
max         1.002000
Name: afactor, dtype: float64


In [21]:
# Save the crosswalk
puma2020_cz20_crosswalk.to_csv('output/puma2020_cz2020_crosswalk.csv', index=False)
print(f'\nSaved: puma2020_cz2020_crosswalk.csv')
print(f'  Rows: {len(puma2020_cz20_crosswalk):,}')
print(f'  PUMA2020s: {puma2020_cz20_crosswalk["puma2020"].nunique():,}')
print(f'  CZ2020s: {puma2020_cz20_crosswalk["cz2020"].nunique():,}')

# Show sample
print(f'\nSample rows:')
print(puma2020_cz20_crosswalk.head(10))


Saved: puma2020_cz2020_crosswalk.csv
  Rows: 3,188
  PUMA2020s: 2,486
  CZ2020s: 598

Sample rows:
  puma2020  cz2020  afactor
0  0100100      10    1.000
1  0100200      10    1.000
2  0100300      10    1.000
3  0100401      10    1.000
4  0100402      10    1.000
5  0100403      10    1.000
6  0100501       8    0.878
7  0100501      10    0.122
8  0100600       8    1.000
9  0100700       8    1.000


In [22]:
puma2020_cz20_crosswalk.to_stata('output/puma2020_cz2020_crosswalk.dta', write_index=False)

In [31]:
puma2020_cz20_crosswalk['puma2020'].str.len().value_counts()

puma2020
7    3188
Name: count, dtype: int64

# Puma2010 to CZ2020

PUMA 2010 → PUMA 2020  → CZ 2020 

In [34]:
# Load IPUMS PUMA 2010 to PUMA 2020 equivalency file
puma2010_to_2020 = pd.read_excel('data/PUMA2010_PUMA2020_crosswalk.xls')

print('IPUMS file structure:')
print(f'Total rows: {len(puma2010_to_2020)}')

# Create full PUMA codes
puma2010_to_2020['puma2010'] = (
    puma2010_to_2020['State10'].astype(str).str.zfill(2) + 
    puma2010_to_2020['PUMA10'].astype(str).str.zfill(5)
)

puma2010_to_2020['puma2020'] = (
    puma2010_to_2020['State20'].astype(str).str.zfill(2) + 
    puma2010_to_2020['PUMA20'].astype(str).str.zfill(5)
)

# Count unique PUMAs
print(f'\nUnique PUMA 2010 codes: {puma2010_to_2020["puma2010"].nunique()}')
print(f'Unique PUMA 2020 codes: {puma2010_to_2020["puma2020"].nunique()}')


IPUMS file structure:
Total rows: 6970

Unique PUMA 2010 codes: 2378
Unique PUMA 2020 codes: 2486


In [35]:
# Use pPUMA10_Pop20 as allocation factor (% of PUMA2010's population in each PUMA2020)
puma2010_to_2020['afactor_10to20'] = puma2010_to_2020['pPUMA10_Pop20'] / 100

# Keep only necessary columns
puma10_to_20 = puma2010_to_2020[['puma2010', 'puma2020', 'afactor_10to20']].copy()

# Verify allocation factors sum to 1 for each PUMA2010
check = puma10_to_20.groupby('puma2010')['afactor_10to20'].sum()
print('PUMA2010 allocation factors:')
print(check.describe())

PUMA2010 allocation factors:
count    2378.000000
mean        0.999999
std         0.000026
min         0.999900
25%         1.000000
50%         1.000000
75%         1.000000
max         1.000100
Name: afactor_10to20, dtype: float64


In [36]:
# Merge with PUMA2020 → CZ2020 crosswalk
puma2010_cz20_crosswalk = pd.merge(
    puma10_to_20,
    puma2020_cz20_crosswalk,
    on='puma2020',
    how='left'
)

print(f'\nNull CZ codes: {puma2010_cz20_crosswalk["cz2020"].isnull().sum()}')


Null CZ codes: 0


In [37]:
# Calculate final allocation factor: PUMA2010 → PUMA2020 → CZ2020
puma2010_cz20_crosswalk['afactor_combined'] = (
    puma2010_cz20_crosswalk['afactor_10to20'] * puma2010_cz20_crosswalk['afactor']
)

# Aggregate to PUMA2010 → CZ2020 level
puma2010_cz20_final = (
    puma2010_cz20_crosswalk.groupby(['puma2010', 'cz2020'])
    .agg({'afactor_combined': 'sum'})
    .reset_index()
)

puma2010_cz20_final = puma2010_cz20_final.rename(columns={'afactor_combined': 'afactor'})
puma2010_cz20_final['afactor'] = puma2010_cz20_final['afactor'].round(3)

# Verify final allocation factors
check_final = puma2010_cz20_final.groupby('puma2010')['afactor'].sum()
print('\nFinal PUMA2010 → CZ2020 allocation factors:')
print(check_final.describe())


Final PUMA2010 → CZ2020 allocation factors:
count    2378.000000
mean        0.999993
std         0.000227
min         0.998000
25%         1.000000
50%         1.000000
75%         1.000000
max         1.002000
Name: afactor, dtype: float64


In [38]:
puma2010_cz20_final.to_csv('output/puma2010_cz2020_crosswalk.csv', index=False)
print(f'  Rows: {len(puma2010_cz20_final):,}')
print(f'  PUMA2010s: {puma2010_cz20_final["puma2010"].nunique():,}')
print(f'  CZ2020s: {puma2010_cz20_final["cz2020"].nunique():,}')

print(puma2010_cz20_final.head(5))

  Rows: 4,861
  PUMA2010s: 2,378
  CZ2020s: 598
  puma2010  cz2020  afactor
0  0100100       5    0.030
1  0100100      10    0.962
2  0100100      12    0.008
3  0100200      10    1.000
4  0100200     474    0.000


In [39]:
puma2010_cz20_final.to_stata('output/puma2010_cz2020_crosswalk.dta', write_index=False)

In [40]:
puma2010_cz20_final['puma2010'].str.len().value_counts()

puma2010
7    4861
Name: count, dtype: int64